# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook explores the FAIR^2 dataset using the `mlcroissant` library. We demonstrate how to load, review, extract, analyze, and visualize structured survey and regression output data, referencing all entities via their `@id` for full traceability.

### Dataset Source
This dataset is defined by a Croissant schema and accessible via this URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.
We will enumerate the record sets defined in the dataset, then for each record set, display their fields and associated column `@id`s. All referencing is via `@id`.

In [ ]:
# Get all record sets from the dataset
record_sets = list(dataset.record_sets())
print(f"Found {len(record_sets)} record sets:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}  | name: {rs.get('name','')}")
    if 'fields' in rs:
        print("  Fields:")
        for field in rs['fields']:
            print(f"    - Field @id: {field['@id']} | name: {field.get('name','')} | dataType: {field.get('dataType','')} | column: {field.get('column','')}")
    else:
        print("  No fields defined.")
    print("")

## 3. Data Extraction
Load each record set's records into a pandas DataFrame for exploration and analysis. Use the record set and field `@id` as discovered above.
We extract all record sets and show sample columns for one selected set.

In [ ]:
# List record set @id values for extraction
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    recs = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(recs)
    dataframes[record_set_id] = df

if len(record_set_ids) > 0:
    sample_record_set_id = record_set_ids[0]
    print(f"Columns in record set {sample_record_set_id}:")
    print(dataframes[sample_record_set_id].columns.tolist())
    print("Sample records:")
    display(dataframes[sample_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply basic filtering, normalization, and grouping on fields in a selected record set, referencing by `@id`.
- Filter by a numeric field (e.g., log likelihood or coefficient field via its `@id`)
- Normalize
- Group by another categorical field

Replace the placeholders below with actual field `@id`s from section 2 as appropriate for the FAIR^2 dataset.

In [ ]:
# Choose a record set and fields by @id
# These values should be filled from section 2 for the FAIR^2 dataset.
if len(record_set_ids) > 0:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]
    # Example field IDs (replace with real @id from your record set):
    numeric_field_id = None
    group_field_id = None

    # Try to guess possible numeric and grouping fields
    for col in df.columns:
        # Guess by name or @id: look for log likelihood, coefficient, or p-value
        if 'log_likelihood' in col.lower() or 'coefficient' in col.lower() or col.lower().startswith('cr:l'):  # e.g. cr:logLikelihood
            numeric_field_id = col
        if ('gender' in col.lower() or 'ward' in col.lower() or 'cr:g' in col.lower()):
            group_field_id = col

    if numeric_field_id is not None:
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records from {rs_id} with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping, if possible
        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped filtered data by {group_field_id}:")
            display(grouped_df.head())
    else:
        print("No suitable numeric field found for EDA. Please update numeric_field_id with an actual field from section 2.")
else:
    print("No record sets loaded for EDA.")

## 5. Visualization
Visualize numeric distributions and relationships between fields, referencing their `@id`.

Below, we use matplotlib to visualize:
- Distribution of a numeric field
- Relationship between a numeric and categorical field

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot numeric field distribution if available
if len(record_set_ids) > 0 and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} in {rs_id}")
    plt.show()
    
    # If grouping field exists, boxplot
    if group_field_id is not None:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Cannot plot distribution: numeric_field_id or record set not available.")

## 6. Conclusion
We loaded the FAIR^2 dataset from its Croissant schema, explored its record sets and fields by their `@id`, extracted all tables, performed initial EDA (filtering, normalization, grouping), and visualized field distributions and relationships. All referencing is by Croissant entity `@id` to ensure full traceability.

You can now build further analyses or modeling using these DataFrames, always relying on explicit `@id` references in FAIR, transparent workflows.